# Урок 12. Разведка данных: группы, сравнения, связи

**Дата:** 10.12.2026 · **60 минут** · 8–9 класс  
**Модуль 3.** Данные (уроки 10–16)  
**Опирается на:** урок 10 (pandas), урок 11 (очистка), урок 5 (`random`), урок 8 (словари).

Данные загружены и почищены. Что дальше? Прежде чем строить модели, инженер **расследует** данные: что в них есть, чем отличаются группы, какие столбцы связаны между собой. Это называется **разведочный анализ данных**, по-английски *EDA* (*exploratory data analysis*). Он помогает не учить модель вслепую и не делать поспешных выводов.

## Цели урока

К концу занятия ты сможешь:

1. Сравнивать группы с помощью `groupby`: средние и количества по категориям.
2. Строить таблицу сочетаний двух категорий (`crosstab`).
3. Измерять связь двух числовых столбцов **корреляцией** и объяснять, почему корреляция — не причина.
4. Помнить о **размере групп** и не делать выводов по трём ученикам.
5. Формулировать **проверяемые гипотезы** и писать короткий отчёт разведки.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–5 | Что такое разведка данных |
| 5–15 | Учебные данные и паспорт |
| 15–35 | `groupby`, `crosstab`, корреляция |
| 35–40 | Гипотезы и ловушки |
| 40–55 | Практика: своё расследование |
| 55–60 | Итог и домашнее задание |

## 1. Разведка данных: вопрос → код → вывод

Разведка — это расследование. Каждый шаг устроен одинаково:

1. **Вопрос:** «У кого выше баллы — у тех, кто занимается больше?»
2. **Код:** таблица, группировка или расчёт.
3. **Вывод с оговоркой:** «На этих 60 учениках — да, заметно. Но групп мало, нужна проверка.»

Хорошая разведка заканчивается не «данные нормальные», а списком **наблюдений**, **гипотез** и **рисков**.

## 2. Учебные данные

Сегодня у нас 60 учеников. Чтобы данные были похожи на настоящие, мы **сгенерируем** их сами с помощью модуля `random` — весь код ниже тебе уже знаком: цикл, словарь, список. Новое только одно:

- `random.gauss(0, 8)` — случайное число около 0, обычно в пределах от −16 до 16. Так мы добавляем **шум**: в жизни балл зависит не только от часов, но и от множества случайностей.
- `random.seed(12)` — «зерно» случайности: с ним при каждом запуске получаются **одни и те же** «случайные» числа. Так у всех в классе одинаковые данные.

In [ ]:
import random
import pandas as pd

def make_students(n, seed):
    """Сгенерировать учебную таблицу учеников."""
    random.seed(seed)
    rows = []
    for i in range(n):
        hours = random.randint(1, 14)
        attendance = min(100, max(20, round(50 + 3 * hours + random.gauss(0, 10))))
        score = min(100, max(0, round(20 + 4 * hours + 0.3 * attendance + random.gauss(0, 8))))
        rows.append({
            "класс": random.choice([8, 9]),
            "кружок": random.choice(["Python", "Роботы", "Игры"]),
            "часы": hours,
            "посещаемость": attendance,
            "сон": round(random.uniform(6, 9.5), 1),
            "балл": score,
        })
    return pd.DataFrame(rows)

df = make_students(60, seed=12)
df["сдал"] = df["балл"] >= 60
df.head()

`random.uniform(6, 9.5)` — случайное дробное число от 6 до 9.5 (часы сна).

Запись `min(100, max(20, …))` держит значение в пределах шкалы: `max(20, x)` не даёт опуститься ниже 20, а `min(100, …)` — подняться выше 100. Так посещаемость не станет 120%, а балл — отрицательным.

Обрати внимание, как мы задали «правило мира»: балл растёт от часов и посещаемости, а **сон и кружок на балл не влияют**. Посмотрим, сможем ли мы это обнаружить разведкой — как будто мы этого не знаем.

## 3. Паспорт и главный показатель

Начинаем, как всегда, с паспорта (урок 10). Потом смотрим на **главный показатель** — то, что нас интересует больше всего. Сегодня это `сдал`: какая доля учеников сдала итоговую работу?

In [ ]:
print("Размер:", df.shape)
print("Пропусков всего:", df.isna().sum().sum())   # первая sum — по столбцам, вторая — общая
df.describe()

In [ ]:
print(df["сдал"].value_counts())
print()
print(df["сдал"].value_counts(normalize=True).round(2))

Если бы сдали 58 из 60, любая «система», которая всегда говорит «сдаст», угадывала бы почти всегда. Поэтому соотношение групп всегда проверяют в начале — это вспомнится на уроках про модели.

## 4. `groupby`: сравниваем группы

`groupby` работает по схеме **«раздели → посчитай → собери»**:

1. **раздели** таблицу на группы по значениям столбца (например, по кружку);
2. **посчитай** что-то в каждой группе (средний балл);
3. **собери** результаты в одну таблицу.

> **Аналогия.** Разложили дневники учеников по стопкам — по кружкам — и в каждой стопке посчитали средний балл.

Это то же самое, что группировка словарём на уроке 8, но одной строкой.

In [ ]:
df.groupby("кружок")["балл"].mean().round(1)

Несколько показателей сразу — через `agg`: в списке перечисляем названия расчётов строками (`"mean"` — среднее, `"median"` — медиана, `"count"` — количество). Обязательно добавляй `count` — **сколько учеников** в группе:

In [ ]:
df.groupby("кружок")["балл"].agg(["mean", "median", "count"]).round(1)

**Загадка.** В «Роботах» средний балл заметно выше. Но когда мы генерировали данные, кружок на балл **никак не влиял**! Откуда тогда разница? Держи загадку в голове — разгадаешь её в задании 1.

Доля сдавших по группам: среднее от столбца `True / False` — это доля `True`.

In [ ]:
df.groupby("класс")["сдал"].agg(["mean", "count"]).round(2)

### Сравниваем тех, кто сдал, и тех, кто нет

Перевернём вопрос: чем отличаются сдавшие от несдавших?

In [ ]:
df.groupby("сдал")[["часы", "посещаемость", "сон"]].mean().round(1)

**Как читать:** если у сдавших среднее заметно выше — столбец **связан** с результатом. Если почти одинаково — связи не видно. Сравни три столбца. Совпадает ли это с «правилом мира», которое мы задали?

## 5. Размер группы важнее красивого числа

«В кружке X сдали 100%!» — а в кружке два ученика. Это не закономерность, а случайность. Посмотрим на маленькие группы:

In [ ]:
df.groupby(["кружок", "класс"])["сдал"].agg(["mean", "count"]).round(2)

Группировка по **двум** столбцам дробит данные на маленькие кусочки — в каждом всего несколько учеников. Чем меньше группа, тем сильнее в ней «гуляет» случайность.

**Правило:** рядом с любым средним по группе смотри `count`. Выводы по группам меньше 10 человек делай очень осторожно.

### `crosstab`: сколько в каждом сочетании

`pd.crosstab` строит таблицу: строки — значения одного столбца, столбцы — другого, в клетках — количество.

In [ ]:
pd.crosstab(df["кружок"], df["сдал"])

## 6. Корреляция: «ходят ли числа вместе?»

**Коэффициент корреляции** — число от −1 до 1, которое показывает, насколько два числовых столбца меняются **вместе**:

| Значение | Смысл | Пример |
|----------|-------|--------|
| около **+1** | один растёт — другой растёт | часы занятий и балл |
| около **0** | связи по прямой не видно | номер ученика и балл |
| около **−1** | один растёт — другой падает | пропуски уроков и балл |

In [ ]:
print("часы и балл:", round(df["часы"].corr(df["балл"]), 2))
print("посещаемость и балл:", round(df["посещаемость"].corr(df["балл"]), 2))
print("сон и балл:", round(df["сон"].corr(df["балл"]), 2))

In [ ]:
# Все пары сразу: таблица корреляций числовых столбцов
df[["часы", "посещаемость", "сон", "балл"]].corr().round(2)

### Корреляция — не причина

Летом растут и продажи мороженого, и число солнечных ожогов. Корреляция высокая, но мороженое ожогов не вызывает — оба зависят от жары. Такую общую причину называют **скрытым фактором**.

В школьных данных: дорогой телефон может коррелировать с баллом — например, через то, что у семьи есть возможность оплатить репетитора. Телефон ум не прибавляет. Если бы мы сделали из этого вывод «купите телефоны — баллы вырастут», это была бы ошибка.

Корреляция говорит «столбцы связаны», но **не говорит, что одно вызывает другое**.

### Случайные корреляции

На маленьких данных корреляция бывает заметной **даже у совершенно случайных** чисел. Проверим: создадим два столбца случайных чисел, никак не связанных между собой, — сначала по 10 значений, потом по 1000.

In [ ]:
random.seed(2)
for n in [10, 1000]:
    a = []
    b = []
    for i in range(n):
        a.append(random.random())
        b.append(random.random())
    noise = pd.DataFrame({"a": a, "b": b})
    print(f"{n} значений: корреляция случайных столбцов = {noise['a'].corr(noise['b']):.2f}")

В `print` здесь новая запись: `{число:.2f}` внутри f-строки — «покажи число с 2 знаками после запятой». Это короче, чем `round`. Похожая запись `{n:4}` выделяет под значение 4 позиции, чтобы столбики в выводе стояли ровно.

На 10 значениях получилась заметная корреляция, хотя связи нет вообще. На 1000 значениях — почти ноль. Вывод: **чем меньше данных, тем легче принять случайность за закономерность**.

## 7. Гипотезы

**Гипотеза** — предположение о связи в данных, которое можно проверить. Хорошая гипотеза:

1. говорит о связи конкретных столбцов;
2. проверяема данными;
3. может оказаться **неверной**.

| Плохо | Хорошо |
|-------|--------|
| «Данные интересные» | «Чем больше часов, тем выше доля сдавших» |
| «Кружок влияет» | «В кружке Python средний балл выше, чем в «Играх»» |
| «Сон точно важен» | «У тех, кто спит больше 8 часов, балл выше» |

Язык осторожности: «на этих данных наблюдается», «похоже», «стоит проверить» — а не «доказано».

### Проверка гипотезы корзинами

Гипотеза: «чем больше часов, тем выше доля сдавших». Разобьём часы на три **корзины** функцией `pd.cut`: она превращает число в категорию по границам `bins`.

In [ ]:
df["часы_группа"] = pd.cut(df["часы"], bins=[0, 4, 9, 14], labels=["1–4", "5–9", "10–14"])
df.groupby("часы_группа", observed=True)["сдал"].agg(["mean", "count"]).round(2)

Границы `bins=[0, 4, 9, 14]` дают корзины «больше 0 и до 4 включительно», «больше 4 и до 9», «больше 9 и до 14». `labels` — подписи корзин. `observed=True` просто говорит pandas показывать только те корзины, которые есть в данных.

## 8. Отчёт разведки

Разведка заканчивается коротким отчётом. Шаблон:

1. **Данные:** сколько строк, какие столбцы, есть ли пропуски.
2. **Главный показатель:** какая доля сдала.
3. **Наблюдения:** 2–3 факта с числами и размерами групп.
4. **Гипотезы:** что, похоже, связано с результатом.
5. **Риски:** мало данных? маленькие группы? возможные скрытые факторы?

## 9. Практика: своё расследование

### Задание 1. Разгадай загадку кружков (5 минут)

Сравни кружки по среднему количеству часов и средней посещаемости. Объясни, почему в «Роботах» баллы выше, хотя кружок на балл не влияет. Как называется такой фактор (раздел 6)?

In [ ]:
# Задание 1

### Задание 2. Сон (7 минут)

Гипотеза: «кто спит больше, тот лучше учится». Проверь её двумя способами: корреляцией и корзинами `pd.cut` по сну (например, `bins=[5, 7, 8, 10]`). Не забудь `count`. Подтвердилась ли гипотеза?

In [ ]:
# Задание 2

### Задание 3. Новые данные — тот же вывод? (7 минут)

Сгенерируй **другую** выборку: `make_students(60, seed=99)`. Повтори сравнение сдавших и несдавших по часам, посещаемости и сну. Совпали ли выводы с первой выборкой? Что совпало, а что изменилось?

In [ ]:
# Задание 3

### Задание 4. Отчёт (5 минут)

Напиши отчёт разведки по шаблону из раздела 8.

**Отчёт разведки:**

1. Данные: …
2. Главный показатель: …
3. Наблюдения: …
4. Гипотезы: …
5. Риски: …

<details>
<summary>Пример решений заданий 1–3</summary>

```python
print(df.groupby("кружок")[["посещаемость", "часы"]].agg(["mean", "count"]).round(1))

print("Корреляция сон—балл:", round(df["сон"].corr(df["балл"]), 2))
df["сон_группа"] = pd.cut(df["сон"], bins=[5, 7, 8, 10], labels=["до 7", "7–8", "больше 8"])
print(df.groupby("сон_группа", observed=True)["сдал"].agg(["mean", "count"]).round(2))

df2 = make_students(60, seed=99)
df2["сдал"] = df2["балл"] >= 60
print(df2.groupby("сдал")[["часы", "посещаемость", "сон"]].mean().round(1))
```
Загадка кружков: в «Роботах» случайно оказались ученики, которые занимаются больше часов, — а часы влияют на балл. Кружок здесь не причина, причина — **скрытый фактор** (часы).

Сон слабо связан с баллом, и на разных выборках эта слабая связь меняет знак — так выглядит случайность. А связь часов и посещаемости с результатом повторяется на любой выборке.
</details>

## Работа с ИИ-помощником

- «Объясни простыми словами, что значит корреляция 0.7.»
- «Придумай ещё два примера, где корреляция есть, а причины нет.»
- «Вот мои наблюдения: … Какие риски в выводах я мог упустить?»

## Мини-глоссарий урока 12

| Термин | Смысл |
|--------|-------|
| **Разведка данных (EDA)** | исследование данных до построения моделей |
| **`groupby`** | раздели на группы → посчитай → собери |
| **`crosstab`** | таблица количеств для сочетаний двух категорий |
| **Корреляция** | число от −1 до 1: насколько столбцы меняются вместе |
| **Скрытый фактор** | общая причина, которая связывает два столбца |
| **Гипотеза** | проверяемое предположение о связи в данных |
| **Шум** | случайные отклонения в данных |

## Итог урока

1. `groupby` сравнивает группы; рядом со средним всегда смотри `count`.
2. Корреляция показывает связь, но **не** причину.
3. Разведка заканчивается наблюдениями, гипотезами и рисками — с осторожными формулировками.

## Домашнее задание (20 минут)

Возьми таблицу «Журнал кружков» из урока 10 (20 учеников) и проведи мини-разведку:

1. Сравни кружки по среднему баллу (с `count`).
2. Найди корреляцию часов и балла, посещаемости и балла.
3. Напиши отчёт по шаблону — 5 пунктов.

## Самопроверка

1. Опиши `groupby` тремя словами.
2. Почему нельзя делать вывод по группе из 3 человек?
3. Что значит корреляция −0.8?
4. Приведи пример, где корреляция есть, а причинной связи нет.
5. Чем гипотеза отличается от вывода?

**Следующий урок (17.12.2026):** визуализация — те же вопросы, но ответы в виде графиков.